# 📐 Aula 17 — t de Student e qui-quadrado no controle de um serviço de pagamentos

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de um serviço de pagamentos online**. O tempo que o serviço leva para aprovar um pagamento (a **latência**) tem distribuição normal, mas o time **não conhece o desvio padrão $\sigma$**: precisa usar o desvio padrão amostral $S$. O notebook segue a ordem da Aula 17:

1. o **problema**: com $S$ no lugar de $\sigma$, o 1,96 da normal deixa de funcionar;
2. a **qui-quadrado** ($\chi^2$): como $S^2$ varia de amostra para amostra;
3. a **t de Student**: a distribuição que substitui a normal padrão quando usamos $S$.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude). Confira antes de usar.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats`**: as **tabelas** da normal (`norm`), da t (`t`) e da qui-quadrado (`chi2`) dentro do computador. Duas funções aparecem neste notebook:
  - `.ppf(p, gl)` dá o número com área $p$ à **esquerda** dele, na distribuição com `gl` graus de liberdade. As tabelas da aula são de **cauda superior** (área $\alpha$ à **direita**), então o valor da tabela é `.ppf(1 - α, gl)`. Exemplo: $t_{0{,}05;15}$ = `t.ppf(0.95, 15)`;
  - `.pdf(x, gl)` dá a altura da curva, para desenhar o gráfico.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a tabela diz.

In [ ]:
import numpy as np                     # calculadora de números e sorteios
import matplotlib.pyplot as plt        # gráficos
from scipy.stats import norm, t, chi2  # as "tabelas"

rng = np.random.default_rng(42)        # sorteador, semente 42
print("Tudo pronto!")

## 1. O problema: com $S$ no lugar de $\sigma$, o 1,96 deixa de funcionar

### A situação
Na verdade (só o notebook sabe), a latência é normal com $\mu = 120$ ms e $\sigma = 15$ ms. O time trabalha com amostras **pequenas**, de **$n = 5$** pagamentos, e não conhece $\sigma$.

### A pergunta
Se o time padronizar a média com o $S$ da amostra, $\dfrac{\bar X - \mu}{S/\sqrt n}$, e usar os limites **−1,96 e 1,96** da normal, que porcentagem das amostras fica fora desses limites? Com $\sigma$ conhecido, seriam **5%**.

### A ferramenta da aula
- Com $\sigma$ **conhecido**, $Z = \dfrac{\bar X - \mu}{\sigma/\sqrt n}$ é normal padrão: 5% das amostras ficam fora de −1,96 a 1,96;
- com $S$ no lugar de $\sigma$, o denominador muda de amostra para amostra, e a padronização segue a **t de Student** com $\nu = n - 1 = 4$ graus de liberdade, que tem caudas mais **pesadas**. Na tabela t, o valor que deixa 95% no meio é $t_{0{,}025;4} = 2{,}776$.

### O que o código faz
1. sorteia **100.000 amostras de 5 latências**;
2. calcula, em cada amostra, o $Z$ (com o $\sigma$ verdadeiro) e o $T$ (com o $S$ da amostra);
3. conta que porcentagem fica fora de −1,96 a 1,96, e que porcentagem do $T$ fica fora de −2,776 a 2,776;
4. desenha o histograma dos valores de $T$ com as curvas da normal e da t por cima.

In [ ]:
mu, sigma, n = 120, 15, 5
amostras = rng.normal(mu, sigma, size=(100_000, n))            # 100.000 amostras de 5 latências
xbarra = amostras.mean(axis=1)
S = amostras.std(axis=1, ddof=1)                               # desvio padrão amostral (divide por n - 1)

Z = (xbarra - mu) / (sigma / np.sqrt(n))                       # com σ conhecido
T = (xbarra - mu) / (S / np.sqrt(n))                           # com S no lugar de σ
t_tab = t.ppf(0.975, n - 1)                                    # t_{0,025;4} = 2,776

print(f"Z fora de ±1,96:    {(np.abs(Z) > 1.96).mean():.1%}   (esperado: 5%)")
print(f"T fora de ±1,96:    {(np.abs(T) > 1.96).mean():.1%}   (usando o valor da normal)")
print(f"T fora de ±{t_tab:.3f}: {(np.abs(T) > t_tab).mean():.1%}   (usando o valor da tabela t com ν = 4)")

x = np.linspace(-5, 5, 400)
plt.hist(T, bins=200, range=(-5, 5), density=True, color="lightsteelblue", edgecolor="none", label="valores de T simulados")
plt.plot(x, norm.pdf(x), color="black", linestyle="--", label="normal padrão")
plt.plot(x, t.pdf(x, n - 1), color="firebrick", linewidth=2, label="t com ν = 4")
plt.xlabel("valor da padronização")
plt.ylabel("Densidade")
plt.title("Com S no lugar de σ, a padronização segue a t, não a normal")
plt.legend()
plt.show()

**Como ler:**
- O $Z$ (com $\sigma$ verdadeiro) fica fora de ±1,96 em **5%** das amostras, como esperado.
- O $T$ (com $S$) fica fora de ±1,96 em cerca de **12%** das amostras — mais que o dobro.
- Com o valor da **tabela t** (2,776), a porcentagem volta para **5%**.
- No gráfico, o histograma de $T$ segue a curva **vermelha** (t), que é mais baixa no centro e mais alta nas pontas que a normal: são as **caudas pesadas** da aula.

O motivo é que $S$ muda de amostra para amostra. Para descrever como $S^2$ varia, a aula usa a qui-quadrado — é a próxima seção.

## 2. A qui-quadrado: quanto $S^2$ pode variar?

### A situação
O contrato com os lojistas (o SLA) diz que o desvio padrão da latência é de **10 ms** ($\sigma^2 = 100$ ms²). A cada dia, o time sorteia **$n = 20$** pagamentos e calcula a variância amostral $S^2$.

### A pergunta
Se o SLA estiver sendo cumprido ($\sigma = 10$), **abaixo de que valor ficam 95% dos $S^2$ observados**? Num dia, a amostra deu $S^2 = 150$ ms²: esse valor está abaixo ou acima do limite?

### A ferramenta da aula
Se a população é normal, então
$$\frac{(n-1)\,S^2}{\sigma^2} \sim \chi^2(n-1).$$
É o mesmo raciocínio do Exemplo 3 da aula (os parafusos).

### A resposta pela conta
- $\nu = 20 - 1 = 19$, e $\dfrac{19\,S^2}{100} \sim \chi^2(19)$;
- 95% abaixo de um valor = 5% na cauda da direita: na tabela, $\chi^2_{0{,}05;19} = 30{,}144$;
- voltando para $S^2$: $\dfrac{19\,S^2}{100} \le 30{,}144 \Rightarrow S^2 \le \dfrac{30{,}144 \times 100}{19} \approx 158{,}65$ ms² (ou $S \le 12{,}6$ ms).

O valor $S^2 = 150$ está **abaixo** do limite.

### O que o código faz
1. calcula o valor da tabela e o limite de $S^2$;
2. **confere a teoria**: sorteia 100.000 amostras de 20 pagamentos de uma normal com $\sigma = 10$ (o SLA sendo cumprido), calcula $S^2$ em cada uma e conta que porcentagem fica abaixo do limite;
3. desenha o histograma dos $S^2$ simulados, com o limite e o valor 150 marcados.

In [ ]:
n, sigma2 = 20, 100
qui2 = chi2.ppf(0.95, n - 1)                                   # χ²_{0,05;19}: área 0,05 à direita
limite = qui2 * sigma2 / (n - 1)

amostras = rng.normal(0, 10, size=(100_000, n))                # SLA cumprido: σ = 10
S2 = amostras.var(axis=1, ddof=1)                              # variância amostral de cada amostra

print(f"Valor da tabela (ν = {n - 1}): {qui2:.3f}")
print(f"Limite de S²: {limite:.2f} ms²   (S até {np.sqrt(limite):.1f} ms)")
print(f"S² abaixo do limite na simulação: {(S2 <= limite).mean():.1%}   (esperado: 95%)")

plt.hist(S2, bins=150, density=True, color="lightsteelblue", edgecolor="none", label="S² simulados (σ = 10)")
plt.axvline(limite, color="firebrick", linewidth=2, label=f"limite = {limite:.1f}")
plt.axvline(150, color="black", linestyle="--", label="S² observado = 150")
plt.xlabel("S² (ms²)")
plt.ylabel("Densidade")
plt.title("Como S² varia entre amostras de 20 pagamentos")
plt.legend()
plt.show()

**Como ler:**
- Com o SLA cumprido, **95%** das amostras de 20 pagamentos têm $S^2$ abaixo de **158,65 ms²** — pela tabela e pela simulação.
- O valor observado, 150, fica **antes** da linha vermelha: está dentro do que se espera quando $\sigma = 10$.
- O histograma é **assimétrico à direita** e não tem valores negativos (a qui-quadrado é uma soma de quadrados), como a aula mostrou. Repare também como $S^2$ varia: mesmo com $\sigma^2 = 100$, 95% das amostras dão $S^2$ entre cerca de 47 e 173 ms².

Decidir formalmente "o SLA foi quebrado ou não" é assunto dos **testes de hipótese** (Aulas 20 a 22).

## 3. Lendo a tabela t

### A situação
O time coletou a latência de **$n = 16$** pagamentos para estudar o tempo médio de aprovação, sem conhecer $\sigma$. A padronização da média com $S$ segue a t com $\nu = 15$.

### A pergunta
1. Que valor de $T$ só é ultrapassado em 5% das amostras? Como ele se compara ao valor da normal padrão ($z = 1{,}645$)?
2. E o valor que deixa **95% no meio** (2,5% em cada cauda), para vários tamanhos de amostra?

### A ferramenta da aula
Com $\sigma$ desconhecido, $T = \dfrac{\bar X - \mu}{S/\sqrt n}$ tem distribuição **t de Student** com $\nu = n - 1$ **graus de liberdade**. Na tabela de cauda superior, $t_{\alpha;\nu}$ é o valor com área $\alpha$ à direita.

### A resposta pela tabela
1. Linha $\nu = 15$, coluna 0,05: $t_{0{,}05;15} = 1{,}753$ (maior que $z = 1{,}645$).
2. Coluna 0,025, uma linha para cada $\nu = n - 1$.

### O que o código faz
1. lê $t_{0{,}05;15}$ na "tabela" da t (`t.ppf(0.95, 15)`);
2. monta uma pequena tabela com $t_{0{,}025;\nu}$ para vários tamanhos de amostra, ao lado do valor da normal (1,96).

In [ ]:
n = 16
print(f"ν = {n - 1}:  t_(0,05;15) = {t.ppf(0.95, n - 1):.3f}   (normal padrão: z = {norm.ppf(0.95):.3f})\n")

print("  n    ν     t_(0,025;ν)    z")
for n_ in (3, 5, 6, 10, 16, 31, 101):
    print(f"{n_:4d} {n_ - 1:4d}        {t.ppf(0.975, n_ - 1):.3f}       {norm.ppf(0.975):.3f}")

**Como ler:**
- Com $\nu = 15$, o valor é **1,753**, um pouco maior que o 1,645 da normal.
- Na tabela, quanto **menor** a amostra, **maior** o valor da t: com 3 observações ele é 4,303, mais que o dobro de 1,96. Com 10 observações é 2,262, o "novo 1,96" do Exemplo 5 da aula. Com 101 observações já é 1,984, quase o valor da normal.
- É o que a seção 1 mostrou na simulação: com $S$, é preciso ir mais longe de zero para deixar os mesmos 5% de fora.

## Para fechar

Neste notebook, o time de pagamentos usou as duas distribuições da Aula 17:

- **qui-quadrado**, com $\nu = n - 1$: é a distribuição de $(n-1)S^2/\sigma^2$, usada para estudar a **variância** — por exemplo, achar abaixo de que valor ficam 95% dos $S^2$;
- **t de Student**, com $\nu = n - 1$: substitui a normal padrão quando usamos $S$ no lugar de $\sigma$. Tem caudas mais pesadas, e por isso valores de tabela maiores, sobretudo em amostras pequenas.

Com a normal, a t e a qui-quadrado nas mãos, a **Aula 18** começa a construir **intervalos de confiança**: faixas de valores que, com uma confiança conhecida, contêm o parâmetro da população.